# 02 · 追踪设置 (ASTRA namelist)

表单模式 (按 namelist 分组, 覆盖手册第 6 章全部 13 个 namelist) 或文本模式 (直接使用现成 .in 文件)。

In [1]:
%run _bootstrap.py

astra-notebook 后端已加载 (v0.1.0)
项目根目录: /Users/yuxinwu/my_projects/astra_notebook
模拟工作目录: /Users/yuxinwu/my_projects/astra_notebook/data/workspace
ASTRA    : /Users/yuxinwu/programs/ASTRA/astra
Generator: /Users/yuxinwu/programs/ASTRA/generator


In [2]:
from astra_tools.widgets.forms import namelist_form
from astra_tools.deck.metadata import summary
print(summary())
print()
print('用法: namelist_form("NEWRUN", only=[...])  # 常用参数')
print('      namelist_form("NEWRUN")              # 全部参数')

  NEWRUN      (6.1) 38 参数
  OUTPUT      (6.2) 33 参数
  SCAN        (6.3) 18 参数
  MODULES     (6.4) 12 参数
  ERROR       (6.5) 57 参数
  CHARGE      (6.6) 27 参数
  APERTURE    (6.7) 21 参数
  WAKE        (6.8) 22 参数
  CAVITY      (6.9) 46 参数
  SOLENOID    (6.10) 12 参数
  QUADRUPOLE  (6.11) 16 参数
  DIPOLE      (6.13) 11 参数
  LASER       (6.14) 26 参数
  INPUT       (7) 66 参数

用法: namelist_form("NEWRUN", only=[...])  # 常用参数
      namelist_form("NEWRUN")              # 全部参数


In [3]:
# 常用追踪参数表单 (逐组生成)
forms, getters = {}, {}
# 基础组: 以可运行的默认值作种子 (纯漂移, 产生全部输出文件)
_basic_seed = {
    "NEWRUN": {"Track_All": True, "Auto_Phase": True,
               "check_ref_part": False, "H_max": 0.001, "H_min": 0.0,
               "Xoff": 0.0, "Yoff": 0.0},
    "OUTPUT": {"ZSTART": 0.0, "ZSTOP": 1.5, "Zemit": 100, "Zphase": 1,
               "RefS": True, "EmitS": True, "PhaseS": True, "SigmaS": True},
}
groups = {
    "NEWRUN": ["Head", "RUN", "Distribution", "Track_All", "Auto_Phase",
               "check_ref_part", "H_max", "H_min", "Xoff", "Yoff", "Toff"],
    "OUTPUT": ["ZSTART", "ZSTOP", "Zemit", "Zphase", "RefS", "EmitS", "PhaseS",
               "SigmaS", "C_EmitS", "Lsub_cor", "Binary", "High_res"],
    "CHARGE": ["LSPCH", "Nrad", "Cell_var", "Nlong_in", "min_grid", "Max_Scale"],
    "CAVITY": ["LEfield", "File_Efield", "C_pos", "Nue", "MaxE", "Phi"],
    "SOLENOID": ["LBfield", "File_Bfield", "S_pos", "MaxB", "S_higher_order"],
    "WAKE": ["LWAKE", "File_Wakefield", "W_pos"],
    "APERTURE": ["LAPERT", "File_APERTURE", "AP_radius"],
}
for gname, only in groups.items():
    wmap, getter = namelist_form(gname, values=_basic_seed.get(gname, {}), only=only)
    forms[gname] = wmap
    getters[gname] = getter
print("各组表单已生成 (基础组已填入可运行默认值)。")

HTML(value='<h3>&amp;NEWRUN</h3>')

HTML(value='<h3>&amp;OUTPUT</h3>')

HTML(value='<h3>&amp;CHARGE</h3>')

HTML(value='<h3>&amp;CAVITY</h3>')

HTML(value='<h3>&amp;SOLENOID</h3>')

HTML(value='<h3>&amp;WAKE</h3>')

HTML(value='<h3>&amp;APERTURE</h3>')

各组表单已生成 (基础组已填入可运行默认值)。


In [4]:
# 写 astra.in (表单模式)
from astra_tools.namelist.write import write_input_deck

blocks = {}
for gname, getter in getters.items():
    # 基础组 (NEWRUN/OUTPUT) 全量写入; 其余组只写用户改动过的参数
    vals = getter(changed_only=(gname not in _basic_seed))
    if vals:
        blocks[gname] = vals
blocks.setdefault("NEWRUN", {})["Distribution"] = "'" + str(SIM_DIR / "bunch.ini") + "'"
write_input_deck(blocks, SIM_DIR / "astra.in",
                 header="generated by astra-notebook (02_astra_setup)")
print("astra.in 已写入, 包含 namelist:", list(blocks))
print()
print((SIM_DIR / "astra.in").read_text())

astra.in 已写入, 包含 namelist: ['NEWRUN', 'OUTPUT']

! generated by astra-notebook (02_astra_setup)
&NEWRUN
  RUN=1,
  Xoff=0,
  Yoff=0,
  Toff=0,
  Track_All=T,
  Auto_Phase=T,
  check_ref_part=F,
  H_max=0.001,
  H_min=0,
  Distribution='/Users/yuxinwu/my_projects/astra_notebook/data/workspace/bunch.ini',
 /

&OUTPUT
  ZSTART=0,
  ZSTOP=1.5,
  Zemit=100,
  Zphase=1,
  Lsub_cor=F,
  RefS=T,
  EmitS=T,
  C_EmitS=F,
  PhaseS=T,
  High_res=F,
  Binary=F,
  SigmaS=T,
 /




**文本模式**: 把现成 .in 文件复制为 `data/workspace/astra.in` 即可。下一步 `03_run.ipynb`。